In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# Helpers
# --------------------------------------------------
def latest_file(patterns):
    candidates = []
    for pattern in patterns:
        candidates.extend(Path(".").glob(pattern))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        raise FileNotFoundError(f"No file found for patterns: {patterns}")
    return max(candidates, key=lambda p: p.stat().st_mtime)

def detect_sep(path: Path) -> str:
    with open(path, "r", encoding="utf-8-sig") as f:
        first = f.readline()
    return ";" if first.count(";") > first.count(",") else ","

def read_auto(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=detect_sep(path))

def periods_to_months(x, period_mode="Q"):
    if period_mode == "Q":
        return x * 3
    if period_mode == "M":
        return x
    if period_mode == "4M":
        return x * 4
    return x

# --------------------------------------------------
# Locate your latest outputs
# --------------------------------------------------
PERIOD_MODE = "Q"

recurrence_path = latest_file([
    "rq2a/recurring_vs_oneoff_topic_features*.csv"
])

growth_path = latest_file([
    "rq2a/growth_decay_topic_features*.csv",
    "rq2a/*growth_decay*features*.csv"
])

long_candidates = []
for pattern in [
    "rq2a/topic_lifecycle_long*.csv",
    "rq2a/*lifecycle_long*.csv"
]:
    long_candidates.extend(Path(".").glob(pattern))
long_candidates = [p for p in long_candidates if p.is_file()]
lifecycle_long_path = max(long_candidates, key=lambda p: p.stat().st_mtime) if long_candidates else None

print("Using recurrence features:", recurrence_path)
print("Using growth features:", growth_path)
print("Using lifecycle long table:", lifecycle_long_path)

recur_df = read_auto(recurrence_path)
growth_df = read_auto(growth_path)
long_df = read_auto(lifecycle_long_path) if lifecycle_long_path else None

# --------------------------------------------------
# Merge
# --------------------------------------------------
df = recur_df.merge(
    growth_df.drop_duplicates("Final"),
    on="Final",
    how="left",
    suffixes=("_recur", "_growth")
)

# unify columns
if "Total_Count_recur" in df.columns:
    df["Total_Count"] = df["Total_Count_recur"]
elif "Total_Count_growth" in df.columns:
    df["Total_Count"] = df["Total_Count_growth"]

if "Lifespan_Periods_recur" in df.columns:
    df["Lifespan_Periods"] = df["Lifespan_Periods_recur"]
elif "Lifespan_Periods_growth" in df.columns:
    df["Lifespan_Periods"] = df["Lifespan_Periods_growth"]

# --------------------------------------------------
# Prevalence concentration
# --------------------------------------------------
df = df.sort_values("Total_Count", ascending=False).reset_index(drop=True)
total_volume = df["Total_Count"].sum()

df["SharePct"] = 100 * df["Total_Count"] / total_volume
df["CumSharePct"] = df["SharePct"].cumsum()

top5_share = df["SharePct"].head(5).sum()
top10_share = df["SharePct"].head(10).sum()
top20_share = df["SharePct"].head(20).sum()

topics_to_50 = int((df["CumSharePct"] < 50).sum() + 1)
topics_to_75 = int((df["CumSharePct"] < 75).sum() + 1)

# persistent vs transient
core_share = 100 * df[df["Recurring_Type"].isin(["Continuous", "Recurring"])]["Total_Count"].sum() / total_volume
transient_share = 100 * df[df["Recurring_Type"].eq("One-Off")]["Total_Count"].sum() / total_volume

# --------------------------------------------------
# Lifespan
# --------------------------------------------------
median_lifespan_periods = float(df["Lifespan_Periods"].median())
mean_lifespan_periods = float(df["Lifespan_Periods"].mean())

median_lifespan_months = periods_to_months(median_lifespan_periods, PERIOD_MODE)
mean_lifespan_months = periods_to_months(mean_lifespan_periods, PERIOD_MODE)

longest_topics = (
    df.sort_values(["Lifespan_Periods", "Total_Count"], ascending=[False, False])
      .head(10)[["Final", "Lifespan_Periods", "Recurring_Type", "Total_Count"]]
      .copy()
)
longest_topics["Lifespan_Months"] = periods_to_months(longest_topics["Lifespan_Periods"], PERIOD_MODE)

shortest_topics = (
    df.sort_values(["Lifespan_Periods", "Total_Count"], ascending=[True, False])
      .head(10)[["Final", "Lifespan_Periods", "Recurring_Type", "Total_Count"]]
      .copy()
)
shortest_topics["Lifespan_Months"] = periods_to_months(shortest_topics["Lifespan_Periods"], PERIOD_MODE)

# --------------------------------------------------
# Growth / decay
# --------------------------------------------------
for col in ["Full_Slope", "Growth_Slope", "Decay_Slope", "Peak_Index", "Last_Index", "Dynamic_Class"]:
    if col not in df.columns:
        df[col] = np.nan

fastest_growing = (
    df.sort_values(["Full_Slope", "Total_Count"], ascending=[False, False])
      .head(10)[["Final", "Full_Slope", "Dynamic_Class", "Total_Count"]]
)

fastest_declining = (
    df.sort_values(["Full_Slope", "Total_Count"], ascending=[True, False])
      .head(10)[["Final", "Full_Slope", "Dynamic_Class", "Total_Count"]]
)

dynamic_counts = (
    df["Dynamic_Class"]
      .fillna("Unknown")
      .value_counts()
      .rename_axis("Dynamic_Class")
      .reset_index(name="Count")
)

# --------------------------------------------------
# Mean time before a topic dies
# only for topics that do NOT survive to final period
# --------------------------------------------------
mean_peak_to_end_periods = np.nan
mean_peak_to_end_months = np.nan
dying_topics_n = 0

if df["Last_Index"].notna().any():
    tmp = df.copy()
    tmp["Peak_Index_num"] = pd.to_numeric(tmp["Peak_Index"], errors="coerce")
    tmp["Last_Index_num"] = pd.to_numeric(tmp["Last_Index"], errors="coerce")
    final_index = int(tmp["Last_Index_num"].max())

    dying = tmp[
        tmp["Last_Index_num"].notna() &
        tmp["Peak_Index_num"].notna() &
        (tmp["Last_Index_num"] < final_index)
    ].copy()

    dying["PeakToEndPeriods"] = dying["Last_Index_num"] - dying["Peak_Index_num"]
    dying = dying[dying["PeakToEndPeriods"] >= 0]

    if len(dying):
        dying_topics_n = int(len(dying))
        mean_peak_to_end_periods = float(dying["PeakToEndPeriods"].mean())
        mean_peak_to_end_months = periods_to_months(mean_peak_to_end_periods, PERIOD_MODE)

# --------------------------------------------------
# Seasonality (exploratory)
# --------------------------------------------------
seasonality_summary = pd.DataFrame()
most_seasonal = pd.DataFrame()

if long_df is not None and {"Period", "Final", "Count"}.issubset(long_df.columns):
    temp = long_df.copy()
    temp["Quarter"] = temp["Period"].astype(str).str.extract(r"(Q[1-4])")
    temp = temp.dropna(subset=["Quarter"])

    quarter_totals = (
        temp.groupby("Quarter", as_index=False)["Count"]
            .sum()
            .rename(columns={"Count": "Total_Count"})
    )
    quarter_totals["SharePct"] = 100 * quarter_totals["Total_Count"] / quarter_totals["Total_Count"].sum()
    seasonality_summary = quarter_totals.copy()

    q_topic = temp.groupby(["Final", "Quarter"], as_index=False)["Count"].sum()
    q_total = q_topic.groupby("Final", as_index=False)["Count"].sum().rename(columns={"Count": "TopicTotal"})
    q_topic = q_topic.merge(q_total, on="Final", how="left")
    q_topic["QuarterSharePct"] = 100 * q_topic["Count"] / q_topic["TopicTotal"]

    most_seasonal = (
        q_topic.groupby("Final", as_index=False)
               .agg(
                   MaxQuarterShare=("QuarterSharePct", "max"),
                   MeanQuarterShare=("QuarterSharePct", "mean"),
                   StdQuarterShare=("QuarterSharePct", "std")
               )
    )
    most_seasonal["SeasonalityRatio"] = most_seasonal["MaxQuarterShare"] / most_seasonal["MeanQuarterShare"]
    most_seasonal = most_seasonal.merge(df[["Final", "Total_Count"]], on="Final", how="left")
    most_seasonal = most_seasonal.sort_values(["SeasonalityRatio", "Total_Count"], ascending=[False, False])

# --------------------------------------------------
# Save
# --------------------------------------------------
out_dir = Path("./rq2d")
out_dir.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([{
    "Top5SharePct": round(top5_share, 2),
    "Top10SharePct": round(top10_share, 2),
    "Top20SharePct": round(top20_share, 2),
    "TopicsTo50Pct": topics_to_50,
    "TopicsTo75Pct": topics_to_75,
    "CoreSharePct": round(core_share, 2),
    "TransientSharePct": round(transient_share, 2),
    "MedianLifespanPeriods": round(median_lifespan_periods, 2),
    "MeanLifespanPeriods": round(mean_lifespan_periods, 2),
    "MedianLifespanMonths": round(median_lifespan_months, 2),
    "MeanLifespanMonths": round(mean_lifespan_months, 2),
    "DyingTopicsN": dying_topics_n,
    "MeanPeakToEndPeriods": None if pd.isna(mean_peak_to_end_periods) else round(mean_peak_to_end_periods, 2),
    "MeanPeakToEndMonths": None if pd.isna(mean_peak_to_end_months) else round(mean_peak_to_end_months, 2),
}])

summary.to_csv(out_dir / "review_summary_stats.csv", sep=";", index=False, encoding="utf-8-sig")
longest_topics.to_csv(out_dir / "review_longest_topics.csv", sep=";", index=False, encoding="utf-8-sig")
shortest_topics.to_csv(out_dir / "review_shortest_topics.csv", sep=";", index=False, encoding="utf-8-sig")
fastest_growing.to_csv(out_dir / "review_fastest_growing_topics.csv", sep=";", index=False, encoding="utf-8-sig")
fastest_declining.to_csv(out_dir / "review_fastest_declining_topics.csv", sep=";", index=False, encoding="utf-8-sig")
dynamic_counts.to_csv(out_dir / "review_dynamic_class_counts.csv", sep=";", index=False, encoding="utf-8-sig")
if not seasonality_summary.empty:
    seasonality_summary.to_csv(out_dir / "review_seasonality_summary.csv", sep=";", index=False, encoding="utf-8-sig")
if not most_seasonal.empty:
    most_seasonal.head(10).to_csv(out_dir / "review_most_seasonal_topics.csv", sep=";", index=False, encoding="utf-8-sig")

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

print("\n=== LONGEST SURVIVING TOPICS ===")
print(longest_topics.head(10).to_string(index=False))

print("\n=== SHORTEST-LIVED TOPICS ===")
print(shortest_topics.head(10).to_string(index=False))

print("\n=== FASTEST GROWING TOPICS ===")
print(fastest_growing.to_string(index=False))

print("\n=== FASTEST DECLINING TOPICS ===")
print(fastest_declining.to_string(index=False))

print("\n=== DYNAMIC CLASS COUNTS ===")
print(dynamic_counts.to_string(index=False))

if not seasonality_summary.empty:
    print("\n=== QUARTER-OF-YEAR SEASONALITY SUMMARY ===")
    print(seasonality_summary.to_string(index=False))

if not most_seasonal.empty:
    print("\n=== MOST SEASONAL TOPICS ===")
    print(most_seasonal.head(10).to_string(index=False))

Using recurrence features: rq2a/recurring_vs_oneoff_topic_features_two_panel_clean_v2.csv
Using growth features: rq2a/growth_decay_topic_features_v6.csv
Using lifecycle long table: None

=== SUMMARY ===
 Top5SharePct  Top10SharePct  Top20SharePct  TopicsTo50Pct  TopicsTo75Pct  CoreSharePct  TransientSharePct  MedianLifespanPeriods  MeanLifespanPeriods  MedianLifespanMonths  MeanLifespanMonths  DyingTopicsN  MeanPeakToEndPeriods  MeanPeakToEndMonths
        53.15          68.67          88.36              5             13         100.0                0.0                   25.0                22.82                  75.0               68.45             3                  9.33                 28.0

=== LONGEST SURVIVING TOPICS ===
                     Final  Lifespan_Periods Recurring_Type  Total_Count  Lifespan_Months
          Forum Reputation                25     Continuous       903883               75
           Online Shopping                25     Continuous       558270           

In [3]:
from pathlib import Path
import pandas as pd

def read_auto(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8-sig") as f:
        first = f.readline()
    sep = ";" if first.count(";") > first.count(",") else ","
    return pd.read_csv(path, sep=sep)

base = Path("./rq2d")
summary = read_auto(base / "review_summary_stats.csv").iloc[0]
longest_topics = read_auto(base / "review_longest_topics.csv")
shortest_topics = read_auto(base / "review_shortest_topics.csv")
fastest_growing = read_auto(base / "review_fastest_growing_topics.csv")
fastest_declining = read_auto(base / "review_fastest_declining_topics.csv")
dynamic_counts = read_auto(base / "review_dynamic_class_counts.csv")

longest_3 = ", ".join(longest_topics["Final"].head(3).tolist())
shortest_3 = ", ".join(shortest_topics["Final"].head(3).tolist())
growing_3 = ", ".join(fastest_growing["Final"].head(3).tolist())
declining_3 = ", ".join(fastest_declining["Final"].head(3).tolist())

results_42_insert = rf"""
\paragraph{{Quantitative prevalence indicators.}}
The prevalence distribution is highly concentrated. The top five topics account for {summary['Top5SharePct']:.2f}\% of total discussion volume, the top ten account for {summary['Top10SharePct']:.2f}\%, and the top twenty account for {summary['Top20SharePct']:.2f}\% of the corpus. Only {int(summary['TopicsTo50Pct'])} topics are required to explain 50\% of all observed activity, and {int(summary['TopicsTo75Pct'])} topics explain 75\%. This confirms that topic prevalence is not broadly distributed across many equally important themes, but is instead organized around a limited set of dominant activities.

This concentration is also reflected in recurrence behavior. Persistent topics, defined here as recurring or continuous topics, account for {summary['CoreSharePct']:.2f}\% of total discussion volume, whereas transient one-off topics account for only {summary['TransientSharePct']:.2f}\%. Thus, the ecosystem is dominated not only by a small number of topics, but specifically by topics that remain active over time.
""".strip()

results_433_insert = rf"""
\paragraph{{Quantitative lifecycle indicators.}}
The lifecycle results provide a more explicit picture of topic persistence and change. The median topic lifespan is {summary['MedianLifespanMonths']:.0f} months ({summary['MedianLifespanPeriods']:.0f} periods), while the mean lifespan is {summary['MeanLifespanMonths']:.1f} months ({summary['MeanLifespanPeriods']:.1f} periods). The longest surviving topics include {longest_3}, whereas the shortest-lived topics include {shortest_3}. These results confirm that most topics remain active for substantial portions of the observation window, while genuinely short-lived themes are comparatively rare.

Growth and decline are similarly uneven but bounded. The strongest positive trends are observed for {growing_3}, while the steepest negative trends are observed for {declining_3}. Among topics that disappear before the final observed period, the mean time from peak activity to last observed activity is {summary['MeanPeakToEndMonths'] if pd.notna(summary['MeanPeakToEndMonths']) else 'N/A'} months. This indicates that even declining topics generally fade over multiple periods rather than disappearing immediately.
""".strip()

discussion_51_add = rf"""
These quantitative indicators reinforce the interpretation that the dark web ecosystem is structurally concentrated. A very small number of topics explain most observed activity, and the topics that dominate are overwhelmingly persistent rather than transient. This suggests that cybercrime platforms repeatedly reproduce a stable set of economically useful functions instead of constantly generating new dominant themes. In other words, the central issue is not topic novelty, but the redistribution of attention within an already established thematic core.
""".strip()

discussion_52_add = rf"""
The lifecycle indicators further show that change is gradual and path-dependent. With a median lifespan of {summary['MedianLifespanMonths']:.0f} months and only limited representation of one-off topics in total discussion volume, most themes persist across years rather than appearing briefly and disappearing. Even for topics that do decline, the average lag between peak activity and disappearance extends across multiple periods. This supports the conclusion that thematic change in cybercrime ecosystems is better understood as a process of gradual reallocation than as abrupt replacement.
""".strip()

lessons_learned = r"""
\subsection{Lessons Learned}

Three lessons emerge from the longitudinal analysis. First, concentration matters more than diversity: a small number of topics explain most of the observed activity. Second, persistence matters more than novelty: recurring and continuous topics dominate the corpus, whereas one-off themes account for only a small share of total volume. Third, longitudinal measurement is necessary to distinguish structural change from short-term fluctuation. Snapshot-based approaches may identify temporary peaks, but they do not capture the much stronger pattern of long-lived and gradually evolving core activities.
""".strip()

print("\n===== INSERT AFTER SECTION 4.2 =====\n")
print(results_42_insert)

print("\n\n===== INSERT AFTER SECTION 4.3.3 =====\n")
print(results_433_insert)

print("\n\n===== ADD TO DISCUSSION 5.1 =====\n")
print(discussion_51_add)

print("\n\n===== ADD TO DISCUSSION 5.2 =====\n")
print(discussion_52_add)

print("\n\n===== INSERT BEFORE 5.4 LIMITATIONS =====\n")
print(lessons_learned)


===== INSERT AFTER SECTION 4.2 =====

\paragraph{Quantitative prevalence indicators.}
The prevalence distribution is highly concentrated. The top five topics account for 53.15\% of total discussion volume, the top ten account for 68.67\%, and the top twenty account for 88.36\% of the corpus. Only 5 topics are required to explain 50\% of all observed activity, and 13 topics explain 75\%. This confirms that topic prevalence is not broadly distributed across many equally important themes, but is instead organized around a limited set of dominant activities.

This concentration is also reflected in recurrence behavior. Persistent topics, defined here as recurring or continuous topics, account for 100.00\% of total discussion volume, whereas transient one-off topics account for only 0.00\%. Thus, the ecosystem is dominated not only by a small number of topics, but specifically by topics that remain active over time.


===== INSERT AFTER SECTION 4.3.3 =====

\paragraph{Quantitative lifecycl

In [5]:
from pathlib import Path
import pandas as pd

def read_auto(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8-sig") as f:
        first = f.readline()
    sep = ";" if first.count(";") > first.count(",") else ","
    return pd.read_csv(path, sep=sep)

base = Path("./rq2d")
summary = read_auto(base / "review_summary_stats.csv").iloc[0]
longest_topics = read_auto(base / "review_longest_topics.csv").head(5)
shortest_topics = read_auto(base / "review_shortest_topics.csv").head(5)
fastest_growing = read_auto(base / "review_fastest_growing_topics.csv").head(5)
fastest_declining = read_auto(base / "review_fastest_declining_topics.csv").head(5)

table_prev = pd.DataFrame([
    ["Top 5 topic share (%)", summary["Top5SharePct"]],
    ["Top 10 topic share (%)", summary["Top10SharePct"]],
    ["Top 20 topic share (%)", summary["Top20SharePct"]],
    ["Topics needed for 50% of activity", int(summary["TopicsTo50Pct"])],
    ["Topics needed for 75% of activity", int(summary["TopicsTo75Pct"])],
    ["Persistent-topic share (%)", summary["CoreSharePct"]],
    ["Transient-topic share (%)", summary["TransientSharePct"]],
], columns=["Indicator", "Value"])

table_life = pd.DataFrame([
    ["Median lifespan (months)", summary["MedianLifespanMonths"]],
    ["Mean lifespan (months)", summary["MeanLifespanMonths"]],
    ["Mean peak-to-end lag (months)", summary["MeanPeakToEndMonths"]],
    ["Longest-lived topics", ", ".join(longest_topics["Final"].tolist())],
    ["Shortest-lived topics", ", ".join(shortest_topics["Final"].tolist())],
    ["Strongest growing topics", ", ".join(fastest_growing["Final"].tolist())],
    ["Strongest declining topics", ", ".join(fastest_declining["Final"].tolist())],
], columns=["Indicator", "Value"])

print("\n=== PREVALENCE TABLE ===\n")
print(table_prev.to_string(index=False))
print("\n=== LIFECYCLE TABLE ===\n")
print(table_life.to_string(index=False))

print("\n=== LATEX PREVALENCE TABLE ===\n")
print(table_prev.to_latex(index=False, escape=True))

print("\n=== LATEX LIFECYCLE TABLE ===\n")
print(table_life.to_latex(index=False, escape=True))


=== PREVALENCE TABLE ===

                        Indicator  Value
            Top 5 topic share (%)  53.15
           Top 10 topic share (%)  68.67
           Top 20 topic share (%)  88.36
Topics needed for 50% of activity   5.00
Topics needed for 75% of activity  13.00
       Persistent-topic share (%) 100.00
        Transient-topic share (%)   0.00

=== LIFECYCLE TABLE ===

                    Indicator                                                                                                                  Value
     Median lifespan (months)                                                                                                                   75.0
       Mean lifespan (months)                                                                                                                  68.45
Mean peak-to-end lag (months)                                                                                                                   28.0
         Longest-lived 